# 005 · Inheritance and super()

> **Syntax › Classes & OOP** · Python 3.14

Single inheritance in Python works much like in C#, with lighter syntax. The big differences: every method is overridable (there is no `virtual`/`override` pair), and Python supports **multiple inheritance**, resolved by a well-defined *Method Resolution Order*.

**In this notebook you will learn:**

- `class Child(Parent)` and calling the parent with `super()`
- overriding — all methods are "virtual", and `typing.override` (3.12+) as the `override` keyword
- `object` as the universal base class, `isinstance` and `issubclass`
- multiple inheritance, the MRO (`__mro__`) and the diamond problem
- mixins as Python's answer to "implementing several interfaces with behaviour"
- `typing.final` to prevent overriding

## C# mental model

| C# | Python |
|---|---|
| `class Dog : Animal` | `class Dog(Animal):` |
| `: base(name)` in the constructor | `super().__init__(name)` inside `__init__` |
| `base.Speak()` | `super().speak()` |
| `virtual` + `override` | every method is overridable; optional `@typing.override` |
| `sealed override` / `sealed class` | `@typing.final` (type checkers only) |
| `System.Object` | `object` |
| single class inheritance + many interfaces | **multiple** inheritance of classes |
| `x is Animal`, `typeof(Dog).IsSubclassOf(...)` | `isinstance(x, Animal)`, `issubclass(Dog, Animal)` |

## Basic inheritance

```csharp
// C#
public class Animal(string name)
{
    public string Name { get; } = name;
    public virtual string Speak() => "...";
}

public class Dog(string name) : Animal(name)
{
    public override string Speak() => "Woof";
}
```

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

    def introduce(self):
        return f"I am {self.name} and I say {self.speak()}"

class Dog(Animal):          # Dog inherits from Animal
    def speak(self):        # override — no keyword needed
        return "Woof"

d = Dog("rex")
print(d.introduce())        # inherited method calls the overridden speak()

`Dog` inherits `__init__` and `introduce` from `Animal`. Because all methods are looked up dynamically on the actual object, `self.speak()` inside `Animal.introduce` calls `Dog.speak` — everything behaves like a C# `virtual` method.

## `super()` — calling the parent

`super()` returns a proxy that forwards to the **next class in the MRO** (for single inheritance, simply the parent). The most common use is chaining `__init__`, the counterpart of `: base(...)`.

⚠️ **Gotcha:** unlike C#, a parent constructor is **not** called automatically. If you define `__init__` in a subclass, you must call `super().__init__(...)` yourself — otherwise the parent's attributes never get set.

```csharp
// C#
public class Puppy : Dog
{
    public int AgeInWeeks { get; }
    public Puppy(string name, int ageInWeeks) : base(name) => AgeInWeeks = ageInWeeks;
    public override string Speak() => base.Speak() + " (squeaky)";
}
```

In [ ]:
class Puppy(Dog):
    def __init__(self, name, age_in_weeks):
        super().__init__(name)                 # C#: : base(name)
        self.age_in_weeks = age_in_weeks

    def speak(self):
        return super().speak() + " (squeaky)"  # C#: base.Speak()

p = Puppy("bit", 8)
print(p.introduce(), "| weeks:", p.age_in_weeks)

What happens if you forget the `super().__init__` call:

In [ ]:
class Cat(Animal):
    def __init__(self, name, indoor):
        self.indoor = indoor              # forgot super().__init__(name)

    def speak(self):
        return "Meow"

c = Cat("tom", True)
try:
    print(c.introduce())
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

You can also call a specific parent explicitly with `Animal.__init__(self, name)` — you'll see this in older code — but `super()` is preferred because it cooperates correctly with multiple inheritance (below).

## `typing.override` — the `override` keyword, for type checkers

Since every method is overridable, a typo in an overriding method's name silently creates a *new* method. Python 3.12 added `@typing.override`: it has no runtime effect (apart from setting `__override__ = True`) but static type checkers report an error if the decorated method does not actually override anything — the same safety the C# `override` keyword gives you.

In [ ]:
from typing import override

class Bird(Animal):
    @override
    def speak(self):
        return "Tweet"

    @override
    def speek(self):            # typo — a type checker flags this; runtime doesn't care
        return "oops"

print(Bird("tweety").introduce())
print(Bird.speak.__override__)

## Preventing overrides — `typing.final`

The counterpart of `sealed` is `@typing.final`, on a class or a method. Like `@override`, it is checked by type checkers only.

In [ ]:
from typing import final

class Payment:
    @final
    def process(self):                 # subclasses should not override this
        self.validate()
        return "processed"

    def validate(self):                # ...but may customise this hook
        pass

class CardPayment(Payment):
    def validate(self):
        print("checking card")

print(CardPayment().process())

## `object` — the root of everything

Every class implicitly inherits from `object` (C#'s `System.Object`). `class Animal:` and `class Animal(object):` are identical in Python 3; the explicit form is a Python 2 leftover.

Use `isinstance` for objects and `issubclass` for classes:

In [ ]:
print(Dog.__bases__)                    # (<class '__main__.Animal'>,)
print(Animal.__bases__)                 # (<class 'object'>,)

print(isinstance(p, Puppy), isinstance(p, Animal), isinstance(p, object))
print(issubclass(Puppy, Animal))        # C#: typeof(Puppy).IsSubclassOf(typeof(Animal))
print(issubclass(Dog, Dog))             # True — a class counts as its own subclass

## Multiple inheritance

C# allows only one base class (plus any number of interfaces). Python lets a class inherit from **several** classes, combining their attributes and methods:

In [ ]:
class Swimmer:
    def swim(self):
        return f"{self.name} swims"

class Flyer:
    def fly(self):
        return f"{self.name} flies"

class Duck(Animal, Swimmer, Flyer):
    def speak(self):
        return "Quack"

duck = Duck("donald")
print(duck.introduce())
print(duck.swim(), "/", duck.fly())
print(isinstance(duck, Flyer))

## Method Resolution Order (MRO)

When several bases define the same method, which one wins? Python computes a single, linear **MRO** for every class (using the *C3 linearisation* algorithm) and searches it left to right. You can inspect it with `__mro__` or `mro()`:

In [ ]:
print([cls.__name__ for cls in Duck.__mro__])

The rules, informally: a class comes before its parents, and parents keep the order you listed them in.

### The diamond problem

The classic diamond: `B` and `C` both derive from `A`, and `D` derives from both. In C++ this can duplicate `A`; in Python, `A` appears **once** in the MRO, and `super()` walks the MRO so each class's method runs exactly once — this is called *cooperative* multiple inheritance.

```
      A
     / \
    B   C
     \ /
      D
```

In [ ]:
class A:
    def greet(self):
        print("A.greet")

class B(A):
    def greet(self):
        print("B.greet")
        super().greet()

class C(A):
    def greet(self):
        print("C.greet")
        super().greet()

class D(B, C):
    def greet(self):
        print("D.greet")
        super().greet()

print([k.__name__ for k in D.__mro__])
D().greet()

Notice that `super()` inside `B` called **`C`**, not `A`. `super()` means "the next class in the MRO of the *actual object*", not "my parent". That is why every class in a cooperative hierarchy should call `super()`, and why `__init__` methods in such hierarchies typically accept `**kwargs` and pass them along.

If the bases are listed in an order that cannot be linearised consistently, Python refuses to create the class:

In [ ]:
try:
    class Bad(A, B):      # A listed before its own subclass B
        pass
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Mixins

A **mixin** is a small class that provides a single, reusable piece of behaviour and is meant to be combined with other classes — never instantiated on its own. Think of it as a C# interface with a default implementation (or an extension method) that can also carry state. By convention, mixin names end in `Mixin` and mixins are listed **before** the main base class.

In [ ]:
import json

class JsonMixin:
    def to_json(self):
        return json.dumps(self.__dict__)

class ReprMixin:
    def __repr__(self):
        fields = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{type(self).__name__}({fields})"

class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

class SmartProduct(JsonMixin, ReprMixin, Product):
    pass

sp = SmartProduct("lamp", 25)
print(sp)
print(sp.to_json())

**Guideline:** prefer composition and simple single inheritance; use multiple inheritance mainly for mixins. Deep or tangled hierarchies are as hard to maintain in Python as in C#. For pure contracts (interfaces), see *Classes & OOP › 006 · Abstract Classes and Protocols*.

## Summary

| Concept | C# | Python |
|---|---|---|
| Inherit | `class Dog : Animal` | `class Dog(Animal):` |
| Call base constructor | `: base(args)` (automatic if parameterless) | `super().__init__(args)` (never automatic) |
| Call base method | `base.Method()` | `super().method()` |
| Overridable | `virtual` / `abstract` only | every method |
| Mark override | `override` (required) | `@typing.override` (optional, 3.12+) |
| Prevent override | `sealed` | `@typing.final` |
| Root type | `System.Object` | `object` |
| Multiple bases | interfaces only | full multiple inheritance |
| Lookup order | single chain | MRO: `Class.__mro__` |
| Type test | `is`, `IsSubclassOf` | `isinstance`, `issubclass` |

## Exercises

1. Create a `Shape` base class with a `name` attribute and an `area()` method returning `0`. Derive `Square(side)` and `Circle(radius)` that call `super().__init__` and override `area()`. Print the area of a list of mixed shapes.
2. Add a `describe()` method to `Shape` that returns `f"{self.name} with area {self.area():.2f}"`, and check it works for both subclasses without changing them.
3. Given `class X: ...`, `class Y(X): ...`, `class Z(X): ...`, `class W(Z, Y): ...`, predict `W.__mro__`, then verify.
4. Write a `TimestampMixin` that adds a `touch()` method setting `self.updated_at` to the current `datetime`, and mix it into a class.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import math
from datetime import datetime

# 1, 2
class Shape:
    def __init__(self, name):
        self.name = name

    def area(self):
        return 0

    def describe(self):
        return f"{self.name} with area {self.area():.2f}"

class Square(Shape):
    def __init__(self, side):
        super().__init__("square")
        self.side = side

    def area(self):
        return self.side ** 2

class Circle(Shape):
    def __init__(self, radius):
        super().__init__("circle")
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

for s in [Square(2), Circle(1)]:
    print(s.describe())

# 3 — W, Z, Y, X, object
class X: ...
class Y(X): ...
class Z(X): ...
class W(Z, Y): ...
print([c.__name__ for c in W.__mro__])

# 4
class TimestampMixin:
    def touch(self):
        self.updated_at = datetime.now()

class Document(TimestampMixin):
    def __init__(self, title):
        self.title = title

doc = Document("notes")
doc.touch()
print(doc.updated_at)
```

</details>